# Creating Model

**GOAL**: For the first round of the World Hockey League tournament, predict the win probability for the home team for 16 matchups.

Since the competition is evaluating on win-probability, accuracy is most likely not going to be the defining metric. Instead, we should focus on log-loss which targets the quality of our predictions.


1) Create a simple linear regression model to be our baseline.
- Split data into training and testing
- Aggregate data to create season-long (only in the data in training) engineering fields
  - This is not going to overfit because we aren't exposing future data
- Fit our model
2) Prevent data leakage
3) Upgrade columns
- Attack vs defense

4) Move to stronger models

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/content/drive/MyDrive/Wharton Data Science Competition 2026/aggregated_plus_engineering.csv")
df.head()

,game_id,team,opponent,toi,is_home,goals_for,goals_against,assists_for,assists_against,shots_for,...,shot_share,xg_share,xg_diff_per60,shot_diff_per60,GP,win_pct,pim_for_per60,pim_against_per60,pim_diff_per60,win
0,game_1,thailand,pakistan,3599.99,1,1,3,2,6,21,...,0.466667,0.506413,0.071500,-3.000008,82,0.609756,16.000044,12.000033,4.000011,0
1,game_10,switzerland,kazakhstan,3600.01,1,4,3,7,4,20,...,0.400000,0.367141,-1.393496,-9.999972,82,0.439024,19.999944,0.000000,19.999944,1
2,game_100,serbia,rwanda,3600.00,1,4,5,7,10,30,...,0.526316,0.548333,0.647200,3.000000,82,0.500000,16.000000,20.000000,-4.000000,0
3,game_1000,brazil,netherlands,3600.03,1,5,0,6,0,32,...,0.542373,0.587009,1.064391,4.999958,82,0.707317,19.999833,11.999900,7.999933,1
4,game_1001,india,morocco,3599.99,1,2,3,4,6,32,...,0.524590,0.478782,-0.306601,3.000008,82,0.597561,16.000044,16.000044,0.000000,0


In [3]:
## We have two rows per game (one for home and away team)
df.sort_values("game_id")

,game_id,team,opponent,toi,is_home,goals_for,goals_against,assists_for,assists_against,shots_for,...,shot_share,xg_share,xg_diff_per60,shot_diff_per60,GP,win_pct,pim_for_per60,pim_against_per60,pim_diff_per60,win
0,game_1,thailand,pakistan,3599.99,1,1,3,2,6,21,...,0.466667,0.506413,0.071500,-3.000008,82,0.609756,16.000044,12.000033,4.000011,0
1312,game_1,pakistan,thailand,3599.99,0,3,1,6,2,24,...,0.533333,0.493587,-0.071500,3.000008,82,0.597561,12.000033,16.000044,-4.000011,1
1313,game_10,kazakhstan,switzerland,3600.01,0,3,4,4,7,30,...,0.600000,0.632858,1.393496,9.999972,82,0.365854,0.000000,19.999944,-19.999944,0
1,game_10,switzerland,kazakhstan,3600.01,1,4,3,7,4,20,...,0.400000,0.367141,-1.393496,-9.999972,82,0.439024,19.999944,0.000000,19.999944,1
2,game_100,serbia,rwanda,3600.00,1,4,5,7,10,30,...,0.526316,0.548333,0.647200,3.000000,82,0.500000,16.000000,20.000000,-4.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1309,game_997,canada,south_korea,3600.01,1,3,5,5,9,22,...,0.392857,0.357605,-1.628995,-11.999967,82,0.439024,17.999950,11.999967,5.999983,0
2622,game_998,uae,switzerland,3600.00,0,2,4,1,2,21,...,0.375000,0.401481,-1.085800,-14.000000,82,0.463415,16.000000,16.000000,0.000000,0
1310,game_998,switzerland,uae,3600.00,1,4,2,2,1,35,...,0.625000,0.598519,1.085800,14.000000,82,0.439024,16.000000,16.000000,0.000000,1
1311,game_999,oman,thailand,3600.01,1,2,3,4,6,29,...,0.547170,0.448806,-0.636498,4.999986,82,0.426829,17.999950,13.999961,3.999989,0


In [4]:
# moving win to the end
# df.insert(len(df.columns)-1, "win", df.pop("win"))

In [5]:
# df.to_csv("/content/drive/MyDrive/Wharton Data Science Competition 2026/aggregated_plus_engineering.csv", index=False)

# Linear Regression

A) Split to training/test

B) Aggregating Training Data

C) Fitting model

## Split to train/test

In [6]:
from sklearn.model_selection import GroupShuffleSplit

# X can be anything; splitter only cares about groups
# X only cares about how many rows and (potentially) the stratification logic
# Sstartification keeps class/label distribution similar in every split in case there is an abnormal split in data (e.g. 70% win and 30% loss)
X = df.index
y = df["win"]
groups = df["game_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

train_df = df.iloc[train_idx].copy()
test_df  = df.iloc[test_idx].copy()

# Sanity checks:
print("Train rows:", len(train_df), "Test rows:", len(test_df))
print("Train unique games:", train_df["game_id"].nunique(),
      "Test unique games:", test_df["game_id"].nunique())

# No game_id overlap:
overlap = set(train_df["game_id"]).intersection(set(test_df["game_id"]))
print("Game_id overlap:", len(overlap))  # should be 0

# Each game should still have 2 rows (team + opponent) inside each split:
print("Train games with !=2 rows:",
      (train_df.groupby("game_id").size() != 2).sum())
print("Test games with !=2 rows:",
      (test_df.groupby("game_id").size() != 2).sum())

print("Train win rate:", train_df["win"].mean())
print("Test win rate:", test_df["win"].mean())


Train rows: 2098 Test rows: 526
Train unique games: 1049 Test unique games: 263
Game_id overlap: 0
Train games with !=2 rows: 0
Test games with !=2 rows: 0
Train win rate: 0.5
Test win rate: 0.5


## Aggregating Training Data

In [7]:
train_df.head().T

,0,1,2,3,4
game_id,game_1,game_10,game_100,game_1000,game_1001
team,thailand,switzerland,serbia,brazil,india
opponent,pakistan,kazakhstan,rwanda,netherlands,morocco
toi,3599.99,3600.01,3600.0,3600.03,3599.99
is_home,1,1,1,1,1
goals_for,1,4,4,5,2
goals_against,3,3,5,0,3
assists_for,2,7,7,6,4
assists_against,6,4,10,0,6
shots_for,21,20,30,32,32


In [8]:
## Aggregating the data and standardize/normalize (per60 and share)

team_agg = train_df.groupby("team", as_index=False).agg(

    GP_train = ("game_id", "nunique"), # groupby number of unique ids
    win_pct_train = ("win", "mean"),
    toi_train = ("toi", "sum"),

    goals_for_train = ("goals_for", "sum"),
    goals_against_train = ("goals_against", "sum"),

    shots_for_train = ("shots_for", "sum"),
    shots_against_train = ("shots_against", "sum"),

    xg_for_train = ("xg_for", "sum"),
    xg_against_train = ("xg_against", "sum"),

    pim_for_train = ("pim_for", "sum"),
    pim_against_train = ("pim_against", "sum")
)

team_agg["minutes_train"] = team_agg["toi_train"]/60
per60_denom = np.maximum(1.0, team_agg["minutes_train"]) # preventing dividing by 0 (just in case)

# ---------------------------- Creating the per60 engineering fields (Not necessarily going to use them all)-------------------------#

# The reason we are using metrics like difference is because that way, we don't have to look at each game individually which leaks the results
team_agg["goal_diff_per60_train"] = ((team_agg["goals_for_train"] - team_agg["goals_against_train"]) / per60_denom) * 60
team_agg["shot_diff_per60_train"] = ((team_agg["shots_for_train"] - team_agg["shots_against_train"]) / per60_denom) * 60
team_agg["xg_diff_per60_train"] = ((team_agg["xg_for_train"] - team_agg["xg_against_train"]) / per60_denom) * 60
team_agg["pim_diff_per60_train"] = ((team_agg["pim_for_train"] - team_agg["pim_against_train"]) / per60_denom) * 60

# ---------------------------------------------------------Share Fields--------------------------------------------------------------#
team_agg["shot_share_train"] = team_agg["shots_for_train"]/np.maximum(1.0, team_agg["shots_for_train"] + team_agg["shots_against_train"])
team_agg["xg_share_train"] = team_agg["xg_for_train"]/np.maximum(1.0, team_agg["xg_for_train"] + team_agg["xg_against_train"])


# --------------------------------------------------------------Pace------------------------------------------------------------------#
team_agg["pace_per60_train"] = ((team_agg["shots_for_train"] + team_agg["shots_against_train"]) / per60_denom)*60

# --------------------------------------------------------Efficiency (rates) -----------------------------------------------------------#
team_agg["xg_per_shot_for_train"]      = team_agg["xg_for_train"] / np.maximum(1.0, team_agg["shots_for_train"])
team_agg["xg_per_shot_against_train"]  = team_agg["xg_against_train"] / np.maximum(1.0, team_agg["shots_against_train"])
team_agg["shooting_pct_train"]         = team_agg["goals_for_train"] / np.maximum(1.0, team_agg["shots_for_train"])
team_agg["save_pct_proxy_train"]       = 1.0 - (team_agg["goals_against_train"] / np.maximum(1.0, team_agg["shots_against_train"]))

# -----------------------------------------------NEW: shrinkage (tune k_shots in CV: 50/100/200/400)-------------------------------------#
k_shots = 200

league_xg_per_shot_for = team_agg["xg_for_train"].sum() / np.maximum(1.0, team_agg["shots_for_train"].sum())
league_xg_per_shot_against = team_agg["xg_against_train"].sum() / np.maximum(1.0, team_agg["shots_against_train"].sum())
league_shooting_pct = team_agg["goals_for_train"].sum() / np.maximum(1.0, team_agg["shots_for_train"].sum())
league_ga_per_sa = team_agg["goals_against_train"].sum () / np.maximum(1.0, team_agg["shots_against_train"].sum())

team_agg["xg_per_shot_for_shrunk_train"] = (
    team_agg["xg_for_train"] + k_shots * league_xg_per_shot_for
    ) / (team_agg["shots_for_train"] + k_shots)

team_agg["xg_per_shot_against_shrunk_train"] = (
    team_agg["xg_against_train"] + k_shots * league_xg_per_shot_against
) / (team_agg["shots_against_train"] + k_shots)

team_agg["shooting_pct_shrunk_train"] = (
    team_agg["goals_for_train"] + k_shots * league_shooting_pct
) / (team_agg["shots_for_train"] + k_shots)

team_agg["save_pct_proxy_shrunk_train"] = 1.0 - (
    (team_agg["goals_against_train"] + k_shots * league_ga_per_sa) /
     (team_agg["shots_against_train"] + k_shots)
)

#------------------------------------------------------Optional Exposure -------------------------------------------------------------#

# lets model implicitly learn to be less confident when strength numbers are based on little data
# reducing overconfidence --> better logloss

team_agg["shots_for_total_train"] = team_agg["shots_for_train"]

# ----------------------------------------------------Creating df with final features------------------------------------------------#

strength_metrics = team_agg[[
    "team",
    "GP_train",

    # share
    "xg_share_train",
    # optional (only if it CV improves)
    # "shot_share_train"

    # efficiency (shrunk) - big for logloss
    "xg_per_shot_for_shrunk_train",
    "xg_per_shot_against_shrunk_train",
    "shooting_pct_shrunk_train",
    "save_pct_proxy_shrunk_train",

    # pace + discipline
    "pace_per60_train",
    "pim_diff_per60_train",

    # optional later (only keep if CV improves)
    # "win_pct_train"

]].copy()

# ------------------------------------------------------Potential Add Back ------------------------------------------------------#

# Start with strength_metrics above and test these one by one

ADD_BACK_CANDIDATES = [
    "shot_share_train",        # sometimes helps alongside xg_share
    "win_pct_train",           # sometimes helps, often hurts calibration
    "xg_diff_per60_train",     # redundant with xg_share (usually no)
    "shot_diff_per60_train",   # redundant with shot_share (usually no)
    "goal_diff_per60_train",   # noisy; often hurts logloss
]

strength_metrics

,team,GP_train,xg_share_train,xg_per_shot_for_shrunk_train,xg_per_shot_against_shrunk_train,shooting_pct_shrunk_train,save_pct_proxy_shrunk_train,pace_per60_train,pim_diff_per60_train
0,brazil,61,0.553158,0.124574,0.100210,0.123159,0.915552,52.690947,1.028743
1,canada,61,0.507209,0.109121,0.110648,0.099729,0.870379,50.284467,-0.974381
2,china,66,0.544462,0.118724,0.103343,0.108447,0.907239,50.568404,-1.817625
3,ethiopia,67,0.473834,0.108430,0.108908,0.120482,0.893547,55.351455,1.436758
4,france,66,0.532510,0.118340,0.113736,0.104143,0.874648,50.320329,-4.073824
5,germany,67,0.491715,0.116263,0.118905,0.120916,0.869464,49.623940,-1.024079
6,guatemala,69,0.518858,0.108077,0.095582,0.101193,0.908087,57.289666,-0.181694
7,iceland,69,0.479536,0.111102,0.108423,0.115241,0.905278,51.741847,1.797172
8,india,67,0.508574,0.102363,0.107803,0.091476,0.911162,53.842833,-2.301831
9,indonesia,59,0.480939,0.103207,0.113030,0.104507,0.902328,50.204163,-3.926034


## `pandas.DataFrame.merge()` (quick)

`merge()` combines two DataFrames by matching values in one or more **key columns** (like a SQL join).

### Basic usage
```python
out = left_df.merge(
    right_df,
    left_on="key_in_left",
    right_on="key_in_right",
    how="left"
)
```

### Key arguments in `merge()`
- **`left_on`**: column(s) in the **left** DataFrame used as the join key.
- **`right_on`**: column(s) in the **right** DataFrame used as the join key.
- **`how`**: join type (which rows to keep):
  - **`"left"`**: keep all rows from the left DataFrame; unmatched right values become `NaN`
  - **`"inner"`**: keep only rows where keys match in both DataFrames
  - **`"right"`**: keep all rows from the right DataFrame; unmatched left values become `NaN`
  - **`"outer"`**: keep all rows from both DataFrames; unmatched values become `NaN`


In [9]:
def merge_team_and_opp_strength(game_df: pd.DataFrame,
                                strength_df: pd.DataFrame,
                                team_col: str = "team",
                                opp_col: str = "opponent",
                                strength_team_key: str = "team",
                                team_prefix: str="team_",
                                opp_prefix: str="opp_") -> pd.DataFrame:
  """
  Merge a per-team strength table onto a per-game table for both the focal team and opponent.

  game_df must be a DataFrame that has columns: team_col and opp_col
  strength_df must have a key column strength_team_key (usually "team") and strengthen feature columns.

  Returns: game_df with team_ and opp_ prefixes appended.
  """

  out = game_df.copy()

  # team-side merge
  team_strength = strength_df.rename(columns={strength_team_key: "team_key"}).add_prefix(team_prefix) # rename to prevent naming confusion
  out = out.merge(team_strength, left_on=team_col, right_on = f"{team_prefix}team_key", how="left")
  out = out.drop(columns=[f"{team_prefix}team_key"])

  # opponent-side merge
  opp_strength = strength_df.rename(columns={strength_team_key: "opp_key"}).add_prefix(opp_prefix)
  out = out.merge(opp_strength, left_on=opp_col, right_on=f"{opp_prefix}opp_key", how="left")
  out = out.drop(columns=[f"{opp_prefix}opp_key"])

  return out

In [10]:
# 2) Create deltas right after merge
def add_deltas (df, cols):
  df = df.copy()
  for c in cols:
    df[f"delta_{c}"] = df[f"team_{c}"] - df[f"opp_{c}"]

  df["min_GP_train"] = np.minimum(df["team_GP_train"], df["opp_GP_train"])
  return df

In [11]:
# 1) MERGE strengths on to new row
train_feat = merge_team_and_opp_strength(train_df, strength_metrics)
test_feat = merge_team_and_opp_strength(test_df, strength_metrics)

use_col = [
    "GP_train",
    "xg_share_train",
    "xg_per_shot_for_shrunk_train",
    "xg_per_shot_against_shrunk_train",
    "shooting_pct_shrunk_train",
    "save_pct_proxy_shrunk_train",
    "pace_per60_train",
    "pim_diff_per60_train",
]

# deltas didn't do much
train_feat = add_deltas(train_feat, use_col)
test_feat  = add_deltas(test_feat, use_col)

# 2) Build Model
feature_cols = (["is_home"] + [f"team_{c}" for c in use_col] + [f"opp_{c}" for c in use_col])

X_train = train_feat[feature_cols]
y_train = train_feat["win"]

X_test = test_feat[feature_cols]
y_test = test_feat["win"]

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])


model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=2000))])

**Note**
The very first scores were the following. I only used regular difference metrics, without shrinkage:

**Accuracy**: 0.5551330798479087

**AUC**: 0.599068947071665

**LogLoss**: 0.6873797279782895

Shrinkage without deltas scores:

**Accuracy**: 0.5437262357414449

**AUC**: 0.6030447165637786

**LogLoss**: 0.6818210095127195

In [13]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score, log_loss

proba = model.predict_proba(X_test)[:,1] # getting probability of win
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc}")
print("AUC:", roc_auc_score(y_test, proba))
print("LogLoss:", log_loss(y_test, proba))

Accuracy: 0.5437262357414449
AUC: 0.6030447165637786
LogLoss: 0.6818210095127195


In [14]:
# coin flip
print("Coin flip:", 0.5)

# home-team always wins baseline (since each game has one home row)
home_pred = (X_test["is_home"] == 1).astype(int)
print("Home baseline:", (home_pred == y_test).mean())


Coin flip: 0.5
Home baseline: 0.5133079847908745


# 5v5 Upgrade

In [15]:
# same code as the top
X = df.index
y = df["win"]
groups = df["game_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

train_df = df.iloc[train_idx].copy()
test_df  = df.iloc[test_idx].copy()

In [16]:
train_df

,game_id,team,opponent,toi,is_home,goals_for,goals_against,assists_for,assists_against,shots_for,...,shot_share,xg_share,xg_diff_per60,shot_diff_per60,GP,win_pct,pim_for_per60,pim_against_per60,pim_diff_per60,win
0,game_1,thailand,pakistan,3599.99,1,1,3,2,6,21,...,0.466667,0.506413,0.071500,-3.000008,82,0.609756,16.000044,12.000033,4.000011,0
1,game_10,switzerland,kazakhstan,3600.01,1,4,3,7,4,20,...,0.400000,0.367141,-1.393496,-9.999972,82,0.439024,19.999944,0.000000,19.999944,1
2,game_100,serbia,rwanda,3600.00,1,4,5,7,10,30,...,0.526316,0.548333,0.647200,3.000000,82,0.500000,16.000000,20.000000,-4.000000,0
3,game_1000,brazil,netherlands,3600.03,1,5,0,6,0,32,...,0.542373,0.587009,1.064391,4.999958,82,0.707317,19.999833,11.999900,7.999933,1
4,game_1001,india,morocco,3599.99,1,2,3,4,6,32,...,0.524590,0.478782,-0.306601,3.000008,82,0.597561,16.000044,16.000044,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2618,game_994,guatemala,france,3600.00,0,4,2,5,2,21,...,0.428571,0.447425,-0.506800,-7.000000,82,0.512195,6.000000,18.000000,-12.000000,1
2619,game_995,mexico,vietnam,3919.98,0,3,4,4,4,23,...,0.433962,0.493945,-0.065296,-6.428604,82,0.463415,11.020464,14.693953,-3.673488,0
2621,game_997,south_korea,canada,3600.01,0,5,3,9,5,34,...,0.607143,0.642395,1.628995,11.999967,82,0.475610,11.999967,17.999950,-5.999983,1
2622,game_998,uae,switzerland,3600.00,0,2,4,1,2,21,...,0.375000,0.401481,-1.085800,-14.000000,82,0.463415,16.000000,16.000000,0.000000,0


In [17]:
test_df

,game_id,team,opponent,toi,is_home,goals_for,goals_against,assists_for,assists_against,shots_for,...,shot_share,xg_share,xg_diff_per60,shot_diff_per60,GP,win_pct,pim_for_per60,pim_against_per60,pim_diff_per60,win
23,game_1019,rwanda,germany,3599.99,1,2,1,3,2,23,...,0.534884,0.515599,0.163100,3.000008,82,0.365854,14.000039,16.000044,-2.000006,1
29,game_1024,uae,new_zealand,3600.00,1,2,5,2,7,21,...,0.355932,0.325520,-2.547200,-17.000000,82,0.463415,16.000000,6.000000,10.000000,0
31,game_1026,serbia,china,3600.01,1,1,7,2,13,28,...,0.482759,0.482539,-0.224199,-1.999994,82,0.500000,9.999972,11.999967,-1.999994,0
32,game_1027,mexico,guatemala,3600.00,1,2,6,4,8,24,...,0.489796,0.517429,0.170700,-1.000000,82,0.463415,10.000000,12.000000,-2.000000,0
43,game_1037,iceland,netherlands,3599.99,1,0,3,0,3,14,...,0.350000,0.324827,-1.282304,-12.000033,82,0.560976,12.000033,6.000017,6.000017,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2609,game_986,panama,morocco,4093.04,0,2,3,1,6,22,...,0.333333,0.328177,-2.166839,-19.349921,82,0.560976,17.590837,5.277251,12.313586,0
2610,game_987,serbia,saudi_arabia,3599.99,0,3,6,6,6,22,...,0.468085,0.427721,-0.689602,-3.000008,82,0.500000,14.000039,8.000022,6.000017,0
2613,game_99,indonesia,peru,3600.00,0,1,5,2,6,22,...,0.423077,0.355077,-1.551600,-8.000000,82,0.500000,14.000000,10.000000,4.000000,0
2616,game_992,pakistan,philippines,3600.01,0,0,1,0,2,22,...,0.478261,0.466047,-0.350699,-1.999994,82,0.597561,15.999956,5.999983,9.999972,0
